# MAS-SHT v10.2 — Experiment Harness

Drives all comparative experiments end-to-end on Google Colab. Six cells, top to bottom:
1. **Setup** — installs deps, mounts Drive, clones repo, loads secrets.
2. **Dataset** — GSM8K test split, configurable sampling.
3. **Runner** — runs all configured baselines + MAS-SHT presets, with checkpointing.
4. **Aggregation** — merges per-system CSVs into one DataFrame.
5. **Statistics** — McNemar tests, comparison table.
6. **Plots** — three-panel figure for the thesis.

Edit `CONFIG` in cell 3 to control which systems run, how many problems, and the seed.

## Cell 1 — Setup

In [ ]:
# === Cell 1 — Setup ============================================================
# Installs all deps, mounts Google Drive, clones the project, loads secrets.
# Safe to re-run: pip skips already-installed pkgs, Drive mount is idempotent.

import os, sys, subprocess

DEPS = [
    'openai>=1.40.0', 'google-generativeai', 'sympy', 'statsmodels',
    'scipy', 'pandas', 'matplotlib', 'tqdm', 'datasets',
    'transformers>=4.44.0', 'accelerate', 'huggingface_hub',
    'python-dotenv', 'tabulate', 'requests',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)

# Mount Google Drive (results + checkpoints persist here).
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — falling back to local filesystem.')

# Clone the project repo (or pull if already present).
REPO_URL = 'https://github.com/USERNAME/MAS_LLM_Thesis.git'
REPO_DIR = '/content/MAS_LLM_Thesis' if IN_COLAB else os.path.expanduser('~/MAS_LLM_Thesis')

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    else:
        subprocess.call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Load API keys from Colab Secrets. Missing keys → warn, do not crash.
def _get_secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

for k in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'HF_API_KEY', 'TOGETHER_API_KEY']:
    v = _get_secret(k)
    if v:
        os.environ[k] = v
    else:
        print(f'WARNING: secret {k} not set — presets that need it will fail.')

# Output directories.
ROOT = '/content/drive/MyDrive/MAS_SHT' if IN_COLAB else os.path.expanduser('~/MAS_SHT')
RESULTS_DIR = f'{ROOT}/results'
CHECKPOINT_DIR = f'{ROOT}/checkpoints'
ARTIFACTS_DIR = f'{ROOT}/artifacts'
for d in [RESULTS_DIR, CHECKPOINT_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Repo: {REPO_DIR}')
print(f'Results: {RESULTS_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## Cell 2 — Dataset

In [ ]:
# === Cell 2 — Dataset ==========================================================
# Loads GSM8K test split. Three sampling modes:
#   {'mode': 'full'}                         all 1319 problems
#   {'mode': 'random',     'n': 100, 'seed': 42}
#   {'mode': 'stratified', 'n': 100, 'seed': 42}   word-count terciles

import re, random
import pandas as pd
from datasets import load_dataset

DATASET_CONFIG = {'mode': 'random', 'n': 50, 'seed': 42}  # safe default for free-tier

def _parse_gold(answer_text: str):
    if '####' in answer_text:
        tail = answer_text.split('####')[-1].strip()
    else:
        tail = answer_text
    nums = re.findall(r'-?\d+(?:,\d+)*(?:\.\d+)?', tail)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(',', ''))
    except ValueError:
        return None

def load_problems(cfg):
    ds = load_dataset('openai/gsm8k', 'main', split='test')
    rng = random.Random(cfg.get('seed', 42))
    items = []
    for i, row in enumerate(ds):
        gold = _parse_gold(row['answer'])
        if gold is None:
            continue
        items.append({
            'problem_id': f'gsm8k_test_{i}',
            'question': row['question'],
            'gold_raw': row['answer'],
            'gold_answer': gold,
            'word_count': len(row['question'].split()),
        })
    mode = cfg.get('mode', 'random')
    if mode == 'full':
        return items
    if mode == 'random':
        rng.shuffle(items)
        return items[: cfg['n']]
    if mode == 'stratified':
        items.sort(key=lambda x: x['word_count'])
        n = len(items)
        thirds = [items[: n // 3], items[n // 3 : 2 * n // 3], items[2 * n // 3 :]]
        per_band = cfg['n'] // 3
        out = []
        for band in thirds:
            rng.shuffle(band)
            out += band[:per_band]
        rng.shuffle(out)
        return out
    raise ValueError(f'Unknown sampling mode: {mode}')

PROBLEMS = load_problems(DATASET_CONFIG)
print(f'Loaded {len(PROBLEMS)} problems (mode={DATASET_CONFIG["mode"]}).')
print('Example:', PROBLEMS[0]['question'][:120], '...')

## Update last version

In [ ]:
import os, glob

# Διαγραφή checkpoint mas_sht_tiny για fresh run
ckpt = os.path.join(CHECKPOINT_DIR, 'mas_sht_tiny.pkl')
if os.path.exists(ckpt):
    os.remove(ckpt)
    print(f"Deleted checkpoint: {ckpt}")

# Προαιρετικά: διαγραφή και των παλιών σπασμένων CSVs
for f in glob.glob(os.path.join(RESULTS_DIR, 'mas_sht_tiny_*.csv')):
    os.remove(f)
    print(f"Deleted: {f}")

In [ ]:
import subprocess
subprocess.call(['git', '-C', '/content/MAS_LLM_Thesis', 'pull', '--ff-only'])

In [ ]:
import importlib, Mas_solver
importlib.reload(Mas_solver)
from Mas_solver import QualityAwarePipeline, UnifiedLLMClient, AgentRole, HETEROGENEOUS_PRESETS, token_budget, _extract_last_number

## Cell 3 — Experiment Runner

In [ ]:
# === Cell 3 — Experiment Runner ================================================
# Iterates every configured (system, preset) over PROBLEMS with checkpointing.
# A timeout in Colab does not lose progress: re-running this cell resumes from
# the latest checkpoint per (system, preset).

import os, time, pickle, traceback
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

import Mas_solver
from Mas_solver import (
    QualityAwarePipeline, UnifiedLLMClient, AgentRole,
    HETEROGENEOUS_PRESETS, token_budget, _extract_last_number,
)
import baselines
from baselines import (
    direct_answer, chain_of_thought, self_consistency, baseline_only,
    BaselineResult,
)

# ------------------------------------------------------------------
# CONFIG — edit me to control the experiment.
# ------------------------------------------------------------------
CONFIG = {
    'baselines_to_run': [],  # παράλειψε τα baselines αν δεν έχεις Groq key τώρα
    'mas_variants': [
        ('mas_sht_tiny',   'tiny_math_homogeneous', True, True),  # Qwen2.5-Math 1.5B
        # ('mas_sht_ds',   'deepseek_distill_1_5b', True, True),  # εναλλακτικά
    ],
    'inter_problem_delay': 1.0,
    'checkpoint_every': 5,
}

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

def _ckpt_path(system_name):
    return os.path.join(CHECKPOINT_DIR, f'{system_name}.pkl')

def _csv_path(system_name):
    return os.path.join(RESULTS_DIR, f'{system_name}_{TIMESTAMP}.csv')

def _load_ckpt(system_name):
    p = _ckpt_path(system_name)
    if os.path.isfile(p):
        try:
            with open(p, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f'  checkpoint load failed for {system_name}: {e}')
    return []

def _save_ckpt(system_name, rows):
    with open(_ckpt_path(system_name), 'wb') as f:
        pickle.dump(rows, f)

def _is_correct(pred, gold):
    if pred is None or gold is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 1e-3
    except (TypeError, ValueError):
        return False

def _row_from_baseline(system, preset, problem, gold_id, gold, br: BaselineResult):
    return {
        'problem_id': gold_id, 'system': system, 'preset': preset,
        'gold': gold, 'predicted': br.answer,
        'correct': _is_correct(br.answer, gold),
        'time_s': br.time_s, 'num_llm_calls': br.num_llm_calls,
        'tokens_estimated': br.tokens_estimated,
        'error_type': br.error_type,
        'timestamp': datetime.now().isoformat(),
    }

def _row_from_mas(system, preset, gold_id, gold, mas_out, time_s):
    mas_block  = mas_out.get('mas', {}) or {}
    sht_block  = mas_out.get('sht', {}) or {}
    siv_block  = mas_out.get('siv', {}) or {}
    base_block = mas_out.get('baseline', {}) or {}
    prog_metrics = mas_block.get('programmer_metrics', {}) or {}

    pred_str = mas_block.get('answer', '')
    pred     = _extract_last_number(str(pred_str))

    base_ans_raw = base_block.get('answer', None)
    base_pred    = _extract_last_number(str(base_ans_raw)) if base_ans_raw is not None else None

    return {
        # ── Identity ──────────────────────────────────────────────────────────
        'problem_id': gold_id,
        'system':     system,
        'preset':     preset,
        'dataset':    'gsm8k_test',

        # ── Correctness ───────────────────────────────────────────────────────
        'gold':             gold,
        'predicted':        pred,
        'correct':          _is_correct(pred, gold),
        'baseline_ans':     base_pred,
        'baseline_correct': _is_correct(base_pred, gold),

        # ── MAS pipeline flags ────────────────────────────────────────────────
        'mas_used_baseline_fallback': mas_block.get('used_baseline_fallback', False),
        'local_hf_fallback':          mas_block.get('local_hf_fallback', False),   # [v10.2]
        'error_type': '' if pred is not None else 'mas_returned_unknown',

        # ── Timing / cost ─────────────────────────────────────────────────────
        'time_s':           time_s,
        'num_llm_calls':    sht_block.get('api_calls_used', 3),
        'tokens_estimated': 0,   # TokenBudget is global, not per-problem for local_hf

        # ── Verification (process-level) ──────────────────────────────────────
        'verification_passed':     prog_metrics.get('verification_passed', True),
        'verification_confidence': prog_metrics.get('verification_confidence', 1.0),
        'solver_agent':            mas_out.get('agents', [None])[0].agent
                                   if mas_out.get('agents') else 'unknown',

        # ── SIV — Layer 1: Execution Audit ────────────────────────────────────
        'siv_execution_audit_passed': siv_block.get('execution_audit_passed'),
        'siv_blueprint_answer':       siv_block.get('blueprint_answer'),
        'siv_execution_rel_error':    siv_block.get('execution_rel_error'),

        # ── SIV — Layer 2: Fault Localisation ────────────────────────────────
        'siv_verified':       siv_block.get('verified'),
        'siv_confidence':     siv_block.get('confidence'),
        'siv_givens_matched': siv_block.get('givens_matched'),
        'siv_givens_total':   siv_block.get('givens_total'),
        'siv_invertible':     siv_block.get('invertible'),
        'siv_failed_givens':  str(siv_block.get('failed_givens', [])),
        'siv_unused_givens':  str(siv_block.get('unused_givens', [])),
        'siv_verifies_translation': False,   # Explicit documented limitation

        # ── SHT ───────────────────────────────────────────────────────────────
        'sht_triggered':      sht_block.get('triggered', False),
        'sht_triage':         sht_block.get('triage_result', 'n/a'),
        'sht_num_candidates': sht_block.get('num_candidates', 0),
        'sht_api_calls':      sht_block.get('api_calls_used', 0),

        # ── Meta ──────────────────────────────────────────────────────────────
        'timestamp': datetime.now().isoformat(),
    }

def _budget_blocked(err: str):
    return ('budget_exceeded' in err.lower()
            or 'rate_limit_daily' in err.lower())

# ------------------------------------------------------------------
# Run baselines (B1-B4)
# ------------------------------------------------------------------
BASELINE_FNS = {
    'b1_direct':         direct_answer,
    'b2_cot':            chain_of_thought,
    'b3_sc5':            lambda c, p: self_consistency(c, p, n=5),
    'b4_baseline_only':  baseline_only,
}

if CONFIG['baselines_to_run']:
    bc = CONFIG['baseline_client']
    baseline_client = UnifiedLLMClient(provider=bc['provider'], model_override=bc['model'])
    for sys_name in CONFIG['baselines_to_run']:
        fn = BASELINE_FNS[sys_name]
        rows = _load_ckpt(sys_name)
        done_ids = {r['problem_id'] for r in rows}
        print(f'[{sys_name}] resuming with {len(done_ids)} done; total {len(PROBLEMS)}')
        budget_hit = False
        for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
            if p['problem_id'] in done_ids:
                continue
            try:
                br = fn(baseline_client, p['question'])
            except Exception as e:
                br = BaselineResult(answer=None, raw=traceback.format_exc()[-500:],
                                    num_llm_calls=0, tokens_estimated=0,
                                    time_s=0.0, error_type='exception')
            rows.append(_row_from_baseline(sys_name, bc['model'], p['question'],
                                           p['problem_id'], p['gold_answer'], br))
            if _budget_blocked(br.error_type) or _budget_blocked(br.raw[:200]):
                print(f'[{sys_name}] Groq daily budget hit at problem {i}. Resume tomorrow.')
                budget_hit = True
            if (i + 1) % CONFIG['checkpoint_every'] == 0 or budget_hit:
                _save_ckpt(sys_name, rows)
            if budget_hit:
                break
            time.sleep(CONFIG['inter_problem_delay'])
        _save_ckpt(sys_name, rows)
        pd.DataFrame(rows).to_csv(_csv_path(sys_name), index=False)
        print(f'[{sys_name}] wrote {len(rows)} rows → {_csv_path(sys_name)}')

# ------------------------------------------------------------------
# Run MAS variants (B5/B6/B7 and any small-model presets)
# ------------------------------------------------------------------
for sys_name, preset, en_siv, en_sht in CONFIG['mas_variants']:
    rows = _load_ckpt(sys_name)
    done_ids = {r['problem_id'] for r in rows}
    print(f'[{sys_name}] preset={preset} siv={en_siv} sht={en_sht} '
          f'resuming with {len(done_ids)} done')
    pipeline = QualityAwarePipeline(
        heterogeneous_preset=preset, use_cache=False,
        enable_siv=en_siv, enable_sht=en_sht,
    )
    budget_hit = False
    for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
        if p['problem_id'] in done_ids:
            continue
        t0 = time.time()
        try:
            mas_out = pipeline.solver.solve(p['question'], str(p['gold_answer']))
        except Exception as e:
            print(f'  exception on {p["problem_id"]}: {e}')
            mas_out = {'mas': {'answer': 'unknown'}, 'siv': {}, 'sht': {}}
        elapsed = time.time() - t0
        rows.append(_row_from_mas(sys_name, preset, p['problem_id'],
                                  p['gold_answer'], mas_out, elapsed))
        if 'budget_exceeded' in str(mas_out).lower() or 'rate_limit_daily' in str(mas_out).lower():
            print(f'[{sys_name}] daily budget hit at problem {i}. Resume tomorrow.')
            budget_hit = True
        if (i + 1) % CONFIG['checkpoint_every'] == 0 or budget_hit:
            _save_ckpt(sys_name, rows)
        if budget_hit:
            break
        time.sleep(CONFIG['inter_problem_delay'])
    _save_ckpt(sys_name, rows)
    pd.DataFrame(rows).to_csv(_csv_path(sys_name), index=False)
    print(f'[{sys_name}] wrote {len(rows)} rows → {_csv_path(sys_name)}')

print('\nAll runs complete.')
print(token_budget.usage_report())

## Cell 4 — Aggregation

In [ ]:
# === Cell 4 — Aggregation ======================================================
# Glob all CSVs in RESULTS_DIR, dedupe (problem_id, system) keeping latest, and
# expose results_dict for the metrics layer.

import glob
import pandas as pd

csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
print(f'Found {len(csv_paths)} CSVs in {RESULTS_DIR}')

frames = [pd.read_csv(p) for p in csv_paths]
if not frames:
    raise SystemExit('No results to aggregate. Run Cell 3 first.')
merged = pd.concat(frames, ignore_index=True)
merged['timestamp'] = pd.to_datetime(merged.get('timestamp'), errors='coerce')
merged = (merged.sort_values('timestamp')
                .drop_duplicates(['problem_id', 'system'], keep='last'))

results_dict = {sys_name: g.reset_index(drop=True)
                for sys_name, g in merged.groupby('system')}
summary = (merged.groupby('system')
                  .agg(n=('problem_id', 'nunique'),
                       accuracy=('correct', 'mean'),
                       avg_calls=('num_llm_calls', 'mean'),
                       avg_time=('time_s', 'mean'))
                  .sort_values('accuracy', ascending=False))
summary

## Cell 5 — Statistical Analysis

In [ ]:
# === Cell 5 — Statistics =======================================================
# Calls evaluation_metrics.{compute_all_metrics, run_mcnemar_tests}.

from evaluation_metrics import compute_all_metrics, run_mcnemar_tests

REFERENCE = 'mas_sht_full'
metrics_df = compute_all_metrics(results_dict, reference_system=REFERENCE)
mcnemar_df = run_mcnemar_tests(results_dict, reference_system=REFERENCE)

metrics_path = os.path.join(ARTIFACTS_DIR, 'comparison_table.csv')
metrics_df.to_csv(metrics_path, index=False)
try:
    md_path = os.path.join(ARTIFACTS_DIR, 'comparison_table.md')
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(metrics_df.to_markdown(index=False, floatfmt='.4f'))
except Exception as e:
    print(f'to_markdown skipped (need tabulate): {e}')
mcnemar_path = os.path.join(ARTIFACTS_DIR, 'mcnemar_results.csv')
mcnemar_df.to_csv(mcnemar_path, index=False)

print('--- Per-system metrics ---')
print(metrics_df.to_string(index=False))
print()
print('--- McNemar (paired) ---')
print(mcnemar_df.to_string(index=False))

## Cell 6 — Plots

In [ ]:
# === Cell 6 — Plots ============================================================
# Three-panel figure for the thesis (accuracy / efficiency / Δ + significance).

from evaluation_metrics import plot_comparison
from IPython.display import Image, display

fig_path = os.path.join(ARTIFACTS_DIR, 'comparison.png')
plot_comparison(metrics_df, mcnemar_df,
                output_path=fig_path,
                reference_system='mas_sht_full',
                title_suffix=f'GSM8K-{DATASET_CONFIG["mode"]}-n{len(PROBLEMS)}')
display(Image(fig_path))
print(f'Saved: {fig_path}')